In [1]:
#Import python modules:
import pandas as pd 
import pandasql as sql
import altair as alt

#Import datasets (I'm adding month dates for later grouping and aggregation):
search_events_raw = pd.read_csv('all_search_events.csv') 
search_events_raw['event_month'] = pd.to_datetime(search_events_raw['event_date']).dt.to_period('M').astype(str)

view_events_raw = pd.read_csv('view_listing_detail_events.csv') 
view_events_raw['event_month'] = pd.to_datetime(view_events_raw['event_date']).dt.to_period('M').astype(str)

reservations_raw = pd.read_csv('reservations.csv')

user_ids = pd.read_csv('amplitude_user_ids.csv')

In [2]:
#There are 3 steps in this converstion funnel. Search > View Listing > Make a reservation
#I want to track users through this funnel and examine "leaks" in converting from a search to a payment. 
#I'm interested in finding the attributes that lead a search to becoming a reservation. 
#It appears that searches and listing views share a "search id" in common. 
#I'll start by cleaning up the individual tables a little:
searches_filter_query = """
select
    event_uuid
    ,event_time
    ,event_date
    ,event_month
    ,merged_amplitude_id as amplitude_user_id
    ,search_id
    ,search_type
    ,search_term
    ,prior_search_id
    ,search_sort
    ,search_term_category
    ,count_results
    ,search_dma
    ,first_attribution_source
    ,first_attribution_channel
from search_events_raw
where 
    is_bot = false 
    and is_host = false 
    and is_neighbor_office = false
"""
searches = sql.sqldf(searches_filter_query, locals())

views_filter_query ="""
select 
    event_uuid
    ,event_time
    ,event_month
    ,merged_amplitude_id as amplitude_user_id 
    ,search_id
    ,listing_id 
    ,click_dma
    ,first_attribution_source
    ,first_attribution_channel
    ,is_listing_reserved
from view_events_raw
where 
    is_bot = false 
    and is_host = false
"""
views = sql.sqldf(views_filter_query,locals()) 

reservations_filter_query ="""
select
    reservation_id
    ,renter_user_id
    ,u.merged_amplitude_id as amplitude_user_id
    ,listing_id
    ,created_at
    ,approved_at
    ,successful_payment_collected_at
    ,dma
from reservations_raw as r
left join user_ids as u on u.user_id = r.renter_user_id
"""

reservations = sql.sqldf(reservations_filter_query,locals())

In [3]:
#Broadly speaking, how many searches get converted to views and how many views get converted to paid reservations on a monthly basis?
import altair as alt
search_conversion_query ="""
with search_sessions as(
    select
        s.search_id
        ,s.amplitude_user_id
        ,s.event_month
        ,s.event_date
        ,s.event_time
        ,lead(s.event_time) over(partition by s.amplitude_user_id order by s.event_time asc) as next_search_event_time
        ,s.first_attribution_source
        ,s.first_attribution_channel
        ,s.search_term
        ,s.search_type
        ,s.search_sort
    from searches as s
)
select
    s.*
    ,v.event_uuid as view_id
    ,v.listing_id
    ,r.reservation_id
    ,r.successful_payment_collected_at
from search_sessions as s
left join views as v on v.search_id = s.search_id
left join reservations as r on 
    r.amplitude_user_id = v.amplitude_user_id
    and r.listing_id = v.listing_id
    and r.created_at between s.event_time and s.next_search_event_time
"""
#In lieu of a key-pair match between a search and a reservation I am windowing the reservations as having happened between searches in order to relate each one to a specific search ID.

search_conversion = sql.sqldf(search_conversion_query, locals())

search_conversion_monthly_query ="""
with view_conversion as(
    select 
        event_month 
        ,search_id 
        ,count(distinct view_id) as view_count
    from search_conversion
    group by 1,2
)
,reservation_conversion as(
    select 
        event_month
        ,search_id
        ,count(distinct reservation_id) as reservation_count 
    from search_conversion
    where successful_payment_collected_at is not null
    group by 1,2
)
,monthly_agg as(
    select
        v.event_month as event_month
        ,count(distinct v.search_id) as total_searches 
        ,sum(
            case
                when v.view_count >= 1
                then 1
                else 0 
            end
        ) as searches_converted_to_views
        ,sum(
            case
                when r.reservation_count >= 1
                then 1 
                else 0 
            end 
        ) as searches_converted_to_reservations
    from view_conversion as v
    left join reservation_conversion as r on r.event_month = v.event_month and r.search_id = v.search_id
    group by 1
)
select 
*
,searches_converted_to_reservations * 1.0 / total_searches * 100 as conversion_rate
from monthly_agg
"""

search_conversion_monthly = sql.sqldf(search_conversion_monthly_query, locals())

search_conversion_monthly_melted = search_conversion_monthly.melt(
    id_vars=['event_month', 'conversion_rate'],
    value_vars=[
        'total_searches',
        'searches_converted_to_views',
        'searches_converted_to_reservations'
    ],
    var_name='metric',
    value_name='count'
)

In [4]:
# Altair viz
sort_order = [
    'total_searches', 
    'searches_converted_to_views',
    'searches_converted_to_reservations'
]

total_conversion_bars = (
    alt.Chart(search_conversion_monthly_melted)
    .mark_bar()
    .encode(
        x=alt.X('event_month:N', title='Month'),
        xOffset=alt.X('metric:N', sort = sort_order),                     
        y=alt.Y('count:Q', title='Count'),
        color=alt.Color('metric:N', title='Metric', sort = sort_order)
        )
    .properties(
        width=1200,  
        height=400   
    )
)

total_conversion_bars

#From these graphs I can see that conversion has remained roughly flat over the 12month period in the data, with conversion from search to paid reservation hovering just under 11%
#I'm curious to know whether there is any obvious winner in terms of the search attribution source that could be better capitalized upon.

alt.Chart(...)

In [5]:
total_conversion_line = (
    alt.Chart(search_conversion_monthly)
    .mark_line(point=True, color='green')
    .encode(
        x=alt.X('event_month:N',title='Month'),
        y=alt.Y('conversion_rate:Q', title='Conversion Rate (%)', axis=alt.Axis(format='.1f')),
        tooltip=['conversion_rate']
    )
    .properties(
        width=1200,  
        height=400   
    )
)
total_conversion_line

alt.Chart(...)

In [6]:
search_conversion_monthly_by_source_query ="""
with conversion as(
    select 
        event_month 
        ,search_id 
        ,first_attribution_channel
        ,count(distinct view_id) as view_count
        ,count(distinct reservation_id) as reservation_count
    from search_conversion
    group by 1,2,3
)
,monthly_agg as(
    select
        event_month as event_month
        ,first_attribution_channel
        ,count(distinct search_id) as total_searches 
        ,sum(
            case
                when view_count >= 1
                then 1
                else 0 
            end
        ) as searches_converted_to_views
        ,sum(
            case
                when reservation_count >= 1
                then 1 
                else 0 
            end 
        ) as searches_converted_to_reservations
    from conversion
    group by 1,2
)
select 
*
,searches_converted_to_reservations * 1.0 / total_searches * 100 as conversion_rate
from monthly_agg
"""

search_conversion_monthly_by_source = sql.sqldf(search_conversion_monthly_by_source_query, locals())
search_conversion_monthly_by_source.to_csv('~/Documents/jupyter/neighbor_search_conversion/search_conversion_monthly_by_source.csv')

In [7]:
conversion_by_source_line = (
    alt.Chart(search_conversion_monthly_by_source)
    .mark_line(point=True, color='green')
    .encode(
        x=alt.X('event_month:N',title='Month'),
        y=alt.Y('conversion_rate:Q', title='Conversion Rate (%)', axis=alt.Axis(format='.1f')),
        color=alt.Color('first_attribution_channel:N', title='Attribution Channel'), 
        tooltip=['first_attribution_channel','conversion_rate']
    )
    .properties(
        width=1200,  
        height=600   
    )
)
conversion_by_source_line

#From these charts we can see that facebook has an appreciably better conversion rate from initial search to paid reservation over recent months, despite having relatively low attribution compared to google. 
#First business recommendation: Analyze current paid ad spend on facebook and consider allocating higher spend toward that channel to capitalize on higher conversion rates. 
#If it is the case that attribution is lower because there are less resources allocated to facebook, then that's an easy lever to pull.
#It might be worthwhile to consider shifting focus from reddit (consistently lower in successful conversions) to facebook instead.
#While the conversion rate from google ads is not as strong it's worth noting that we have extremely high attribution from that source, leading me to believe that it is still a key top-of-funnel source.

alt.Chart(...)

In [8]:
conversion_by_source_bar = (
    alt.Chart(search_conversion_monthly_by_source)
    .mark_bar()
    .encode(
        x=alt.X('event_month:N', title='Month'),
        xOffset=alt.X('first_attribution_channel:N', title='Attribution Channel'),
        y=alt.Y('total_searches:Q', title='Number of Searches'),
        color=alt.Color('first_attribution_channel:N', title='Attribution Channel')
    )
    .properties(width=800, height=400)
)
conversion_by_source_bar

alt.Chart(...)

In [9]:
#From the above I also notice that there is a significant number of searches that do not turn into views. I'd like to know the cause of that "leak"

searches_with_view_counts_query ="""
with view_counts as(
    select 
        s.search_id
        ,count(distinct v.event_uuid) as view_count
    from searches as s
    left join views as v on v.search_id = s.search_id 
    group by 1
)
select 
    s.*
    ,v.view_count
from searches as s
left join view_counts as v on v.search_id = s.search_id 
"""

searches_with_view_counts = sql.sqldf(searches_with_view_counts_query, locals())

#There is no obvious trend to me here with any specific configurable field of the search (search type, category,attribution channel etc.) that leads to a view count of 0

#In that case, how many searches do users execute in one session? Do they make more searches after one that leads to zero views? I.E. do they keep trying to find something to fit their needs?

search_sessions_query ="""
with search_counts_by_day as(
    select 
        event_date
        ,amplitude_user_id
        ,count(distinct search_id) as total_daily_searches
    from searches
    group by 1,2
)
,searches_ranked as(
    select 
        sv.event_date
        ,sv.amplitude_user_id
        ,sv.search_id
        ,sv.view_count
        ,row_number() over(partition by sv.event_date, sv.amplitude_user_id order by sv.event_time asc) as search_rownum
        ,sc.total_daily_searches
    from searches_with_view_counts as sv 
    left join search_counts_by_day as sc on sc.amplitude_user_id = sv.amplitude_user_id and sc.event_date = sv.event_date
)
select 
    view_count
    ,search_rownum
    ,total_daily_searches
    ,count(distinct concat(event_date,amplitude_user_id)) as sessions
    from searches_ranked
where view_count = 0
group by 1,2
"""

search_sessions = sql.sqldf(search_sessions_query,locals())
print(search_sessions)

#Since we don't have any session info here I've proxied the date of the search 
#Any searches from a given user that occur on the same day are considered to be part of the same daily "session"

#From this query alone it's easy to see that the overwhelming majority of the time a user executes a search and doesn't find what they're looking for in the results (does not view any listings)
#They will not execute another search in that daily session 
#So how can we keep inbound users from "bouncing" after one search with no views? 
#One possible solution could be a pop-up that appears if the user attempts to navigate away after a single search with no views. 
#This could feature Gen AI in some capacity. 
#For example a "search assistant" that automatically generates a refined set of recommended listings in the user's area with links to view each.
#If the user is not inclined to refine their search, we can attempt to help them. 

   view_count  search_rownum  total_daily_searches  sessions
0           0              1                     1     42135
1           0              2                     2       625
2           0              3                     3         7


In [10]:
#I noticed that a not insignificant number of views are on listings that are already reserved. 
#I'm curious how often this happens and what user behavior looks like after viewing a reserved listing. 
explore_query ="""
with reserved_views as (
    select
        event_month
        ,count(event_uuid) as view_count
        ,sum(case
            when is_listing_reserved = true 
            then 1 
            else 0
        end) as reserved_listing_view_count
    from views
    group by 1
)
select 
event_month
,reserved_listing_view_count * 1.0 / view_count 
from reserved_views
"""

query_df = sql.sqldf(explore_query, locals())

# query_df.to_csv('~/Documents/jupyter/neighbor_search_conversion/explore_results.csv')
print(query_df) 

#From this query I can see that 9-10% of views every month are on listings that are already reserved. Not insignificant.
#Investigation should be done into why this happens. Are there stale searches? 
#In any case, 9-10% of views categorically can't be converted into listings. 
#What do users do after running into a reserved listing? Do they modify their search and try again with different terms or dates?
#Unfortunately I don't have user requested dates available in the search data, but I can check for user behavior after finding a reserved listing. 


   event_month  reserved_listing_view_count * 1.0 / view_count
0      2023-01                                        0.097678
1      2023-02                                        0.102389
2      2023-03                                        0.100920
3      2023-04                                        0.099304
4      2023-05                                        0.099163
5      2023-06                                        0.099469
6      2023-07                                        0.099445
7      2023-08                                        0.100133
8      2023-09                                        0.097264
9      2023-10                                        0.105019
10     2023-11                                        0.098996
11     2023-12                                        0.097523


In [11]:
reserved_listing_conversion_query="""
with search_counts_by_day as(
    select 
        event_date
        ,amplitude_user_id
        ,count(distinct search_id) as total_daily_searches
    from searches
    group by 1,2
)
,searches_ranked as(
    select 
        s.event_date
        ,s.event_time
        ,s.amplitude_user_id
        ,s.search_id
        ,row_number() over(partition by s.event_date, s.amplitude_user_id order by s.event_time asc) as search_rownum
        ,lead(s.event_time) over(partition by s.amplitude_user_id order by s.event_time asc) as next_search_event_time
        ,sc.total_daily_searches
    from searches as s 
    left join search_counts_by_day as sc on sc.amplitude_user_id = s.amplitude_user_id and sc.event_date = s.event_date
)
,views_ranked as(
    select
        s.*
        ,v.amplitude_user_id
        ,v.event_uuid as view_id
        ,v.listing_id
        ,v.is_listing_reserved 
        ,row_number() over(partition by s.event_date, v.search_id order by v.event_time asc) as view_rownum
    from searches_ranked as s
    left join views as v on v.search_id = s.search_id
)
,searches_to_reservations as(
select 
*
,r.reservation_id
,r.successful_payment_collected_at
from views_ranked as vr
left join reservations as r on  
    r.amplitude_user_id = vr.amplitude_user_id
    and r.listing_id = vr.listing_id
    and r.created_at between vr.event_time and vr.next_search_event_time
    and r.successful_payment_collected_at is not null
)
,reserved_listing_encounters as (
    select 
        search_id
        ,count(distinct view_id) as total_views
        ,sum(is_listing_reserved) as reserved_listing_views 
        ,count(distinct reservation_id) as reservations
    from searches_to_reservations
    group by 1
    having (count(distinct view_id) is not null and count(distinct view_id) > 0)
)
,search_conversion as (
    select 
        search_id
        ,total_views
        ,case 
            when reserved_listing_views = 1
            then 1 
            else 0 
        end as encountered_1_reserved_listing 
        ,case 
            when reserved_listing_views > 1
            then 1
            else 0 
        end as encountered_multiple_reserved_listings
        ,case 
            when reservations > 0 
            then 1 
            else 0 
        end as successful_reservation
    from reserved_listing_encounters
)
select 
*,
case 
    when encountered_1_reserved_listing = 1
    then "Encountered 1 Reserved Listing"
    when encountered_multiple_reserved_listings = 1
    then "Encountered Multiple Reserved Listings"
    else "Encountered No Reserved Listings" 
end as category
from search_conversion
"""

reserved_listing_conversion = sql.sqldf(reserved_listing_conversion_query, locals())
# ranked_searches_daily.to_csv('~/Documents/jupyter/neighbor_search_conversion/explore_results.csv')
print(reserved_listing_conversion)


                                  search_id  total_views  \
0      00008a27-805a-4152-a2d0-7119294e1818            1   
1      000251d9-bd63-4dfb-93a0-3da6a6e8563e            1   
2      000734ef-b788-475d-b3c3-30ddee9db1c6            1   
3      0007a454-ce44-4685-ab3d-486d10df77f7            2   
4      00098212-15fe-4f91-b926-f1aacb92be9d            2   
...                                     ...          ...   
42700  fff6814c-3e99-4a06-8561-dfd0bc892a52            3   
42701  fff72443-b71f-4594-837c-c38148c75f26            2   
42702  fff8b68c-a138-4bbd-ba3a-08ad43434acd            1   
42703  fffb6a46-02fc-4321-bcb3-dc415069139d            1   
42704  fffd60af-93f0-43cc-b910-16ff9faf9742            1   

       encountered_1_reserved_listing  encountered_multiple_reserved_listings  \
0                                   0                                       0   
1                                   0                                       0   
2                                   

In [12]:
summary = (
    reserved_listing_conversion.groupby('category')
      .agg(total_searches=('search_id','count'),
           successful_reservations=('successful_reservation','sum'))
      .reset_index()
)

summary['pct_success'] = summary['successful_reservations'] / summary['total_searches'] * 100

# Altair bar chart
chart = alt.Chart(summary).mark_bar().encode(
    x=alt.X('category:N', title='Search Category'),
    y=alt.Y('pct_success:Q', title='Successful Reservation Rate (%)'),
    tooltip=['total_searches', 'successful_reservations', 'pct_success']
).properties(
    title='Percentage of Searches Ending in Successful Reservation by Category'
)

chart
#It appears that there is not significant drop-off in users that hit reserved listings, indicating that it isn't a significant deterrent. 
#These users are probably still shopping and will look for other, non-reserved listings. 
#However, users view an average of 3 listings per search. It would likely boost conversion rates to only show them unreserved listings (or feature is available as a toggle in the search bar)

alt.Chart(...)